# Model Inference — საუკეთესო მოდელი Model Registry-დან

**დავალების მოთხოვნა:** test set-ზე პროგნოზი საუკეთესო მოდელით; მოდელი უნდა იყოს **Model Registry**-ში დარეგისტრირებული, **იქიდან ჩამოტვირთული** და ისე გამოძახებული `predict`.

ნაბიჯები:
1. ყველა არქიტექტურის შედარება (`best_runs.json` — თითო notebook თავის საუკეთესოს იწერდა იქ)
2. საუკეთესოს დარეგისტრირება **W&B Model Registry**-ში
3. მოდელის ჩამოტვირთვა **პირდაპირ registry-დან** და გაშვება **raw** test set-ზე → `submission_best_model.csv`
4. ბონუსი: ყველა არქიტექტურის submission + ensemble blend-ები

ყველა შენახული pipeline **raw** test.csv-ის სვეტებზე (`Store, Dept, Date, IsHoliday`) მუშაობს — preprocessing არ სჭირდება.

In [1]:
!pip install -q wandb xgboost lightgbm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import wandb

wandb.login()  # API key: https://wandb.ai/authorize

WANDB_PROJECT = 'ML-Final'  # გუნდის საერთო პროექტი
GROUP = 'Model_Registry'  # "ექსპერიმენტი" ამ არქიტექტურისთვის — ყველა run ამ ჯგუფში ერთიანდება
NB_VERSION = 'final'  # ჩაიწერება run-ების config-ში, რომ wandb-ზე ვერსიები გაიფილტროს

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dshon23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']  # ზუსტად ის სვეტები, რაც raw test.csv-შია
VAL_CUTOFF = '2012-08-17'  # იგივე time split, რაც გუნდმა EDA_Preprocessing-ში გამოიყენა


def wmae(y_true, y_pred, is_holiday):
    """Competition metric: MAE, სადაც სადღესასწაულო კვირებს წონა 5 აქვთ."""
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## 1. არქიტექტურების შედარება

ყველა შედეგი ერთსა და იმავე holdout-ზეა (`>= 2012-08-17`, WMAE) — პირდაპირ შედარებადია.

In [5]:
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
with open(reg_path) as f:
    best_runs = json.load(f)

print(f'{len(best_runs)} architectures found')  # უნდა იყოს 8: XGB, LGBM, DLinear, NBEATS, PatchTST, Prophet, ARIMA, SARIMA
comparison = (pd.DataFrame(best_runs).T
                .sort_values('val_wmae')
                .rename_axis('architecture'))
comparison

8 architectures found


,artifact,val_wmae
architecture,,
XGBoost,ML-Final/xgboost_pipeline:latest,1451.361785
NBEATS,ML-Final/nbeats_pipeline:latest,1463.00219
PatchTST,ML-Final/patchtst_pipeline:latest,1499.25118
LightGBM,ML-Final/lightgbm_pipeline:latest,1508.410617
DLinear,ML-Final/dlinear_pipeline:latest,1602.686594
Prophet,ML-Final/prophet_pipeline:latest,1831.404617
ARIMA,ML-Final/arima_pipeline:latest,1914.194822
SARIMA,ML-Final/sarima_pipeline:latest,2137.259343


## 2. საუკეთესო მოდელის დარეგისტრირება W&B Model Registry-ში

ჯერ ვცდით ახალი სტილის Registry-ს (`wandb-registry-model/...`), ჩავარდნისას — ძველი სტილის Model Registry-ს (`<entity>/model-registry/...`).

In [6]:
REGISTRY_NAME = 'walmart-best-model'

best_name = comparison.index[0]
best = best_runs[best_name]
print(f'Best architecture: {best_name} | val WMAE: {best["val_wmae"]:,.2f}')

run = wandb.init(project=WANDB_PROJECT, group='Model_Registry', name='Register_Best_Model',
                 job_type='registry',
                 config={'architecture': best_name, 'val_wmae': best['val_wmae'],
                         'artifact': best['artifact']})
art = run.use_artifact(best['artifact'])
try:
    art.link(f'wandb-registry-model/{REGISTRY_NAME}')
    REGISTRY_PATH = f'wandb-registry-model/{REGISTRY_NAME}:latest'
except Exception as e:
    print('new-style registry unavailable, falling back to legacy:', e)
    entity = wandb.Api().default_entity
    art.link(f'{entity}/model-registry/{REGISTRY_NAME}')
    REGISTRY_PATH = f'{entity}/model-registry/{REGISTRY_NAME}:latest'
run.finish()
print('Registered at:', REGISTRY_PATH)

Best architecture: XGBoost | val WMAE: 1,451.36


Registered at: wandb-registry-model/walmart-best-model:latest


## 3. ჩატვირთვა Registry-დან და პროგნოზი raw test-ზე

მოდელი მოდის **მხოლოდ registry-ის მისამართიდან** (არა პირდაპირ ტრენინგის artifact-იდან) — ზუსტად ისე, როგორც production-ში იქნებოდა.

In [7]:
import cloudpickle

run = wandb.init(project=WANDB_PROJECT, group='Model_Registry', name='Inference_From_Registry',
                 job_type='inference', config={'registry_path': REGISTRY_PATH})

reg_art = run.use_artifact(REGISTRY_PATH)
model_dir = reg_art.download()
pkl_file = [f for f in os.listdir(model_dir) if f.endswith('.pkl')][0]
with open(os.path.join(model_dir, pkl_file), 'rb') as f:
    best_model = cloudpickle.load(f)
print('Loaded from registry:', pkl_file)

# pipeline პირდაპირ RAW test set-ზე
test_pred = best_model.predict(test[RAW_COLS])

submission = pd.DataFrame({
    'Id': (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
           + '_' + test['Date'].dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': test_pred,
})
submission.to_csv('submission_best_model.csv', index=False)
sub_art = wandb.Artifact('best_model_submission', type='submission',
                         metadata={'architecture': best_name})
sub_art.add_file('submission_best_model.csv')
run.log_artifact(sub_art)
run.finish()
submission.head()

wandb:   1 of 1 files downloaded.  
/tmp/ipykernel_1253/2067742159.py:10: UserWarning: [15:31:38] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:443: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  best_model = cloudpickle.load(f)
/tmp/ipykernel_1253/2067742159.py:10: UserWarning: [15:31:38] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  best_model = cloudpickle.load(f)
/tmp/ipykernel_1253/2067742159.py:10: UserWarning: [15:31:38] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  best_model = cloudpickle.load(f)


Loaded from registry: xgboost_pipeline.pkl


,Id,Weekly_Sales
0,1_1_2012-11-02,40161.117188
1,1_1_2012-11-09,19513.494141
2,1_1_2012-11-16,20051.917969
3,1_1_2012-11-23,21035.513672
4,1_1_2012-11-30,20908.894531


## 4. ბონუსი: ყველა არქიტექტურის პროგნოზი + Ensemble blend-ები

- NBEATS/PatchTST/ARIMA/SARIMA-ს Kaggle submission ჯერ არ ჰქონდათ — აქ გენერირდება
- **blend_trees** (0.6·XGB + 0.4·LGBM) — უკვე გატესტილი Kaggle-ზე: 2 865.2 (აქამდე საუკეთესო)
- **blend_top3** (0.5·XGB + 0.3·NBEATS + 0.2·LGBM) — NBEATS განსხვავებული ოჯახია (holiday-exog ნეირონული ქსელი), მისი შეცდომები tree მოდელებისგან დამოუკიდებელია → მეტი დივერსიფიკაცია
- **blend_diverse** (0.4·XGB + 0.25·NBEATS + 0.2·LGBM + 0.15·PatchTST) — ოთხი მოდელი სამი ოჯახიდან

In [8]:
api = wandb.Api()


def load_pipeline(artifact_ref):
    art = api.artifact(artifact_ref)
    d = art.download()
    pkl = [f for f in os.listdir(d) if f.endswith('.pkl')][0]
    with open(os.path.join(d, pkl), 'rb') as f:
        return cloudpickle.load(f)


preds = {}
for name, info in best_runs.items():
    try:
        preds[name] = load_pipeline(info['artifact']).predict(test[RAW_COLS])
        print(f'{name}: OK')
    except Exception as e:
        print(f'{name}: skipped ({e})')

blends = {
    'blend_trees': 0.6 * preds['XGBoost'] + 0.4 * preds['LightGBM'],
    'blend_top3': 0.5 * preds['XGBoost'] + 0.3 * preds['NBEATS'] + 0.2 * preds['LightGBM'],
    'blend_diverse': (0.4 * preds['XGBoost'] + 0.25 * preds['NBEATS']
                      + 0.2 * preds['LightGBM'] + 0.15 * preds['PatchTST']),
}

ids = (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
       + '_' + test['Date'].dt.strftime('%Y-%m-%d'))

for name, p in {**preds, **blends}.items():
    fname = f'submission_{name.lower()}.csv'
    pd.DataFrame({'Id': ids, 'Weekly_Sales': p}).to_csv(fname, index=False)
    print('saved', fname)

print('\nKaggle-ზე ასატვირთი (score ჯერ არ გვაქვს): nbeats, patchtst, arima, sarima, '
      'blend_top3, blend_diverse')

wandb:   1 of 1 files downloaded.  
/tmp/ipykernel_1253/2656051272.py:9: UserWarning: [15:31:47] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:443: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  return cloudpickle.load(f)
/tmp/ipykernel_1253/2656051272.py:9: UserWarning: [15:31:47] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  return cloudpickle.load(f)
/tmp/ipykernel_1253/2656051272.py:9: UserWarning: [15:31:47] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  return cloudpickle.load(f)


XGBoost: OK


wandb:   1 of 1 files downloaded.  


LightGBM: OK


wandb:   2 of 2 files downloaded.  


...

Prophet: OK


wandb:   1 of 1 files downloaded.  


ARIMA: OK


wandb:   1 of 1 files downloaded.  


SARIMA: OK
saved submission_xgboost.csv
saved submission_lightgbm.csv
saved submission_dlinear.csv
saved submission_nbeats.csv
saved submission_patchtst.csv
saved submission_prophet.csv
saved submission_arima.csv
saved submission_sarima.csv
saved submission_blend_trees.csv
saved submission_blend_top3.csv
saved submission_blend_diverse.csv

Kaggle-ზე ასატვირთი (score ჯერ არ გვაქვს): nbeats, patchtst, arima, sarima, blend_top3, blend_diverse


## შეჯამება

- საუკეთესო არქიტექტურა დარეგისტრირდა Registry-ში და submission **registry-დან ჩამოტვირთული** მოდელით დაგენერირდა — დავალების მოთხოვნა შესრულებულია.
- Kaggle-ზე ატვირთეთ და PROGRESS_NOTES-ში ჩაიწერეთ: `submission_nbeats.csv`, `submission_patchtst.csv`, `submission_blend_top3.csv` (blend_top3 სავარაუდოდ ახალი საუკეთესო იქნება).
- README-ს საბოლოო ცხრილი ამ notebook-ის შედარების ცხრილს + Kaggle score-ებს დაეყრდნობა.